# Davis2017

[https://doi.org/10.1098/rstb.2016.0381](https://doi.org/10.1098/rstb.2016.0381)

In [ ]:
import string

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from mxlpy import Simulator, make_protocol
from mxlpy.parallel import parallelise

from mxlmodels import get_davis2017

## Figure 3

In [ ]:
def fig3_sim(settings: dict):
    m = get_davis2017()
    m.update_variables(settings["y0"])
    m.update_parameter("P_K", settings["P_K"])

    s = Simulator(m)
    s.simulate_protocol(settings["protocol"], time_points_per_step=int(1e4))

    return s.get_result().unwrap_or_err().get_combined()


fig3_prtc = make_protocol(
    [
        (2.5 * 60, {"PPFD": 0}),
        (5 * 60, {"PPFD": 300}),
        (10 * 60, {"PPFD": 0}),
    ]
)

settings = {
    "0": {
        "y0": {
            "QA_red": 1.6680442448215374e-05,
            "PQH_2": 0.01996349863700503,
            "pH_lumen": 6.455422734896244,
            "Dpsi": 0.014062169675170863,
            "K_lu": 0.023317976223034567,
            "PC_ox": 1.2976782989183835,
            "Zx": 0.07361151276346266,
            "singO2": 0.2874820769637723,
            "P700_ox": 0,
            "Fd_red": 0,
            "NADPH_st": 0,
            "LEF": 0,
            "ATP_made": 0,
        },
        "P_K": get_davis2017().get_parameter_values()["P_K"] * 0,
        "protocol": fig3_prtc,
    },
    "1": {
        "y0": {
            "QA_red": 5.511496362768013e-06,
            "PQH_2": 0.006610917859794705,
            "pH_lumen": 6.455489733520293,
            "Dpsi": 0.014066213897372587,
            "K_lu": 0.023314403407362544,
            "PC_ox": 1.6965704506492556,
            "Zx": 0.0735694070105809,
            "singO2": 0.38023965479069094,
            "P700_ox": 0,
            "Fd_red": 0,
            "NADPH_st": 0,
            "LEF": 0,
            "ATP_made": 0,
        },
        "P_K": get_davis2017().get_parameter_values()["P_K"] * 1,
        "protocol": fig3_prtc,
    },
    "10": {
        "y0": {
            "QA_red": 1.6680442448215374e-05,
            "PQH_2": 0.01996349863700503,
            "pH_lumen": 6.455422734896244,
            "Dpsi": 0.014062169675170863,
            "K_lu": 0.023317976223034567,
            "PC_ox": 1.2976782989183835,
            "Zx": 0.07361151276346266,
            "singO2": 0.2874820769637723,
            "P700_ox": 0,
            "Fd_red": 0,
            "NADPH_st": 0,
            "LEF": 0,
            "ATP_made": 0,
        },
        "P_K": get_davis2017().get_parameter_values()["P_K"] * 10,
        "protocol": fig3_prtc,
    },
}

fig3_res = {
    key: val
    for key, val in parallelise(
        fig3_sim, [(key, vals) for key, vals in settings.items()]
    )
}
fig3_res = pd.concat(fig3_res.values(), keys=fig3_res.keys())
fig3_res.index.names = ["P_K_fold", "time"]

In [ ]:
fig3, axs = plt.subplots(nrows=5, ncols=3, figsize=(11, 15), sharex=True)

plot_stylings = {
    "Dpsi": {"color": "#1e1bfe", "ls": "-", "label": r"$\Delta \Psi$"},
    "delta_pH_inVolts": {"color": "#fa323a", "ls": "--", "label": r"$\Delta$pH"},
    "pmf": {"color": "#0d6806", "ls": "--", "label": r"$pmf$"},
    "k_b6f": {"color": "#0506fc", "ls": "--"},
    "pH_lumen": {"color": "#fb0f1d", "ls": "-"},
    "NPQ": {"color": "#107207", "ls": "--"},
    "K_lu": {"color": "#191919", "ls": "-"},
    "QA_red": {"color": "#246b18", "ls": "--"},
    "vPSII_recomb": {"color": "#fc030f", "ls": "-"},
    "LEF": {"color": "#157814", "ls": "--"},
    "singO2": {"color": "#fb131f", "ls": "-"},
}

for i in range(len(axs[0])):
    res = fig3_res.xs(fig3_res.index.levels[0][i])
    res.index = res.index / 60

    for var in ["Dpsi", "delta_pH_inVolts", "pmf"]:
        axs[0, i].plot(res[var] - res[var].iloc[0], **plot_stylings.get(var, {}))

    axs[1, i].plot(res["k_b6f"] * 5, **plot_stylings.get("k_b6f", {}))
    ax_pH = axs[1, i].twinx()
    ax_pH.plot(res["pH_lumen"], **plot_stylings.get("pH_lumen", {}))

    axs[2, i].plot(res["NPQ"], **plot_stylings.get("NPQ", {}))
    ax_K = axs[2, i].twinx()
    ax_K.plot(res["K_lu"] * 1e2, **plot_stylings.get("K_lu", {}))

    axs[3, i].plot(res["QA_red"] / 2, **plot_stylings.get("QA_red", {}))
    ax_PSII = axs[3, i].twinx()
    ax_PSII.plot(res["vPSII_recomb"] / 4, **plot_stylings.get("vPSII_recomb", {}))

    axs[4, i].plot(res["LEF"] * 1e-4, **plot_stylings.get("LEF", {}))
    ax_O2 = axs[4, i].twinx()
    ax_O2.plot(res["singO2"] * 1e-3, **plot_stylings.get("singO2", {}))

    axs[0, i].set_ylim([-0.05, 0.125])
    axs[0, i].set_yticks(np.linspace(0, 0.1, 3))
    axs[0, 0].set_ylabel(r"$\Delta$V)")

    axs[1, i].set_ylim([0.75, 2.5])
    axs[1, i].set_yticks(np.linspace(1.0, 2.5, 4))
    axs[1, 0].set_ylabel(
        r"$b_6f$ rate constant (s$^{-1}$)",
        color=plot_stylings.get("k_b6f", {}).get("color", "black"),
    )
    axs[1, i].spines["left"].set_color(
        plot_stylings.get("k_b6f", {}).get("color", "black")
    )
    axs[1, i].spines["right"].set_visible(False)
    axs[1, i].tick_params(
        axis="y", colors=plot_stylings.get("k_b6f", {}).get("color", "black")
    )
    axs[1, 0].text(
        0,
        1.01,
        r"1 $\times 10^2$",
        transform=axs[1, 0].transAxes,
        color=plot_stylings.get("k_b6f", {}).get("color", "black"),
    )
    ax_pH.set_ylim([5.8, 6.5])
    ax_pH.set_yticks(np.linspace(5.8, 6.4, 4))
    ax_pH.set_ylabel(
        "Lumen pH", color=plot_stylings.get("pH_lumen", {}).get("color", "black")
    )
    ax_pH.spines["right"].set_color(
        plot_stylings.get("pH_lumen", {}).get("color", "black")
    )
    ax_pH.spines["left"].set_visible(False)
    ax_pH.tick_params(
        axis="y", colors=plot_stylings.get("pH_lumen", {}).get("color", "black")
    )

    axs[2, i].set_ylim([-0.1, 4])
    axs[2, i].set_yticks(np.linspace(0, 4, 5))
    axs[2, 0].set_ylabel(
        r"NPQ ($q_\mathrm{E}$)",
        color=plot_stylings.get("NPQ", {}).get("color", "black"),
    )
    axs[2, i].spines["left"].set_color(
        plot_stylings.get("NPQ", {}).get("color", "black")
    )
    axs[2, i].spines["right"].set_visible(False)
    axs[2, i].tick_params(
        axis="y", colors=plot_stylings.get("NPQ", {}).get("color", "black")
    )
    ax_K.set_ylim([0.25, 2.5])
    ax_K.set_yticks(np.linspace(0.5, 2.5, 5))
    ax_K.set_ylabel(
        "lumen $K^+$ (M)", color=plot_stylings.get("K_lu", {}).get("color", "black")
    )
    ax_K.spines["right"].set_color(plot_stylings.get("K_lu", {}).get("color", "black"))
    ax_K.spines["left"].set_visible(False)
    ax_K.tick_params(
        axis="y", colors=plot_stylings.get("K_lu", {}).get("color", "black")
    )
    if i == len(axs[0]) - 1:
        ax_K.text(
            1,
            1.01,
            r"1 $\times 10^{-2}$",
            transform=ax_K.transAxes,
            color=plot_stylings.get("K_lu", {}).get("color", "black"),
            ha="right",
        )

    axs[3, i].set_ylim([-0.05, 0.39])
    axs[3, i].set_yticks(np.linspace(0, 0.3, 4))
    axs[3, 0].set_ylabel(
        r"$Q^-_\mathrm{A}$", color=plot_stylings.get("QA_red", {}).get("color", "black")
    )
    axs[3, i].spines["left"].set_color(
        plot_stylings.get("QA_red", {}).get("color", "black")
    )
    axs[3, i].spines["right"].set_visible(False)
    axs[3, i].tick_params(
        axis="y", colors=plot_stylings.get("QA_red", {}).get("color", "black")
    )
    ax_PSII.set_ylim([-1.25, 12.5 + 1.25])
    ax_PSII.set_yticks(np.linspace(0, 12.5, 6))
    ax_PSII.set_ylabel(
        r"$^1\mathrm{O}_2$ (s$^-1$)",
        color=plot_stylings.get("vPSII_recomb", {}).get("color", "black"),
    )
    ax_PSII.spines["right"].set_color(
        plot_stylings.get("vPSII_recomb", {}).get("color", "black")
    )
    ax_PSII.spines["left"].set_visible(False)
    ax_PSII.tick_params(
        axis="y", colors=plot_stylings.get("vPSII_recomb", {}).get("color", "black")
    )

    axs[4, i].set_ylim([0, 6])
    axs[4, i].set_yticks(np.linspace(0, 5, 6))
    axs[4, 0].set_ylabel(
        "LEF cumulative", color=plot_stylings.get("LEF", {}).get("color", "black")
    )
    axs[4, i].spines["left"].set_color(
        plot_stylings.get("LEF", {}).get("color", "black")
    )
    axs[4, i].spines["right"].set_visible(False)
    axs[4, i].tick_params(
        axis="y", colors=plot_stylings.get("LEF", {}).get("color", "black")
    )
    axs[4, 0].text(
        0,
        1.01,
        r"1 $\times 10^4$",
        transform=axs[4, 0].transAxes,
        color=plot_stylings.get("LEF", {}).get("color", "black"),
    )
    ax_O2.set_ylim([0, 4])
    ax_O2.set_yticks(np.linspace(0, 4, 5))
    ax_O2.set_ylabel(
        r"$^1\mathrm{O}_2$ (cumulative)",
        color=plot_stylings.get("singO2", {}).get("color", "black"),
    )
    ax_O2.spines["right"].set_color(
        plot_stylings.get("singO2", {}).get("color", "black")
    )
    ax_O2.spines["left"].set_visible(False)
    ax_O2.tick_params(
        axis="y", colors=plot_stylings.get("singO2", {}).get("color", "black")
    )
    if i == len(axs[0]) - 1:
        ax_O2.text(
            1,
            1.01,
            r"1 $\times 10^3$",
            transform=ax_O2.transAxes,
            color=plot_stylings.get("singO2", {}).get("color", "black"),
            ha="right",
        )

    axs[4, i].set_xlabel("time (min)")
    axs[4, i].set_xlim([0, 17.5])
    axs[4, i].set_xticks(np.linspace(0, 15, 4))

    for j in range(len(axs)):
        ax = axs[j, i]

        if i != 2:
            for sec_ax in [ax_pH, ax_K, ax_PSII, ax_O2]:
                sec_ax.set_yticks([])
                sec_ax.set_ylabel("")

        if i != 0:
            ax.set_yticks([])
            ax.set_ylabel("")

        ax.text(
            0.85,
            0.15,
            f"${string.ascii_lowercase[i]}.{j + 1}$",
            ha="center",
            va="center",
            transform=ax.transAxes,
            bbox=dict(boxstyle="circle,pad=0.4", edgecolor="black", facecolor="white"),
        )

        ax.add_patch(
            plt.Rectangle(
                xy=(fig3_prtc.index.total_seconds()[0] / 60, ax.get_ylim()[0]),
                width=fig3_prtc.index.total_seconds()[1] / 60
                - fig3_prtc.index.total_seconds()[0] / 60,
                height=ax.get_ylim()[1] - ax.get_ylim()[0],
                facecolor="red",
                alpha=0.15,
            )
        )

axs[0, 0].legend(frameon=False)

plt.tight_layout()

## Figure 4

In [ ]:
def sin_wave(t: float):
    if t <= 0:
        return 0
    frequency_factor = (2 * np.pi) / (2 * 60 * 60)
    return 300 * np.sin(frequency_factor * t)


def fig4_sim(settings: dict):
    m = get_davis2017()
    m.update_variables(settings["y0"])
    m.update_parameter("P_K", settings["P_K"])

    if settings["protocol"] == "sin":
        m.remove_parameter("PPFD")
        m.add_derived(name="PPFD", fn=sin_wave, args=["time"])
        return (
            Simulator(m)
            .simulate(60 * 60, steps=int(1e4))
            .get_result()
            .unwrap_or_err()
            .get_combined()
        )

    prtc = make_protocol(
        [(2.5 * 60, {"PPFD": 0}), (2.5 * 60, {"PPFD": 300})] * int(60 / (2.5 * 2))
    )
    return (
        Simulator(m)
        .simulate_protocol(prtc, time_points_per_step=10)
        .get_result()
        .unwrap_or_err()
        .get_combined()
    )


settings = {
    "normal_sin": {
        "y0": {
            "QA_red": 5.511496362768013e-06,
            "PQH_2": 0.006610917859794705,
            "pH_lumen": 6.455489733520293,
            "Dpsi": 0.014066213897372587,
            "K_lu": 0.023314403407362544,
            "PC_ox": 1.6965704506492556,
            "Zx": 0.0735694070105809,
            "singO2": 0.38023965479069094,
            "P700_ox": 0,
            "Fd_red": 0,
            "NADPH_st": 0,
            "LEF": 0,
            "ATP_made": 0,
        },
        "P_K": get_davis2017().get_parameter_values()["P_K"] * 1,
        "protocol": "sin",
    },
    "fast_sin": {
        "y0": {
            "QA_red": 1.6680442448215374e-05,
            "PQH_2": 0.01996349863700503,
            "pH_lumen": 6.455422734896244,
            "Dpsi": 0.014062169675170863,
            "K_lu": 0.023317976223034567,
            "PC_ox": 1.2976782989183835,
            "Zx": 0.07361151276346266,
            "singO2": 0.2874820769637723,
            "P700_ox": 0,
            "Fd_red": 0,
            "NADPH_st": 0,
            "LEF": 0,
            "ATP_made": 0,
        },
        "P_K": get_davis2017().get_parameter_values()["P_K"] * 10,
        "protocol": "sin",
    },
    "normal_square": {
        "y0": {
            "QA_red": 5.511496362768013e-06,
            "PQH_2": 0.006610917859794705,
            "pH_lumen": 6.455489733520293,
            "Dpsi": 0.014066213897372587,
            "K_lu": 0.023314403407362544,
            "PC_ox": 1.6965704506492556,
            "Zx": 0.0735694070105809,
            "singO2": 0.38023965479069094,
            "P700_ox": 0,
            "Fd_red": 0,
            "NADPH_st": 0,
            "LEF": 0,
            "ATP_made": 0,
        },
        "P_K": get_davis2017().get_parameter_values()["P_K"] * 1,
        "protocol": "square",
    },
    "fast_square": {
        "y0": {
            "QA_red": 1.6680442448215374e-05,
            "PQH_2": 0.01996349863700503,
            "pH_lumen": 6.455422734896244,
            "Dpsi": 0.014062169675170863,
            "K_lu": 0.023317976223034567,
            "PC_ox": 1.2976782989183835,
            "Zx": 0.07361151276346266,
            "singO2": 0.2874820769637723,
            "P700_ox": 0,
            "Fd_red": 0,
            "NADPH_st": 0,
            "LEF": 0,
            "ATP_made": 0,
        },
        "P_K": get_davis2017().get_parameter_values()["P_K"] * 10,
        "protocol": "square",
    },
}

fig4_res = {
    key: val
    for key, val in parallelise(
        fig4_sim, [(key, vals) for key, vals in settings.items()]
    )
}
fig4_res = pd.concat(fig4_res.values(), keys=fig4_res.keys())
fig4_res.index.names = ["sim_ver", "time"]

In [ ]:
fig4, axs = plt.subplots(nrows=2, ncols=4, figsize=(8, 6), sharex=True)

plot_stylings = {
    "Dpsi": {"color": "#1e1bfe", "ls": "-", "label": r"$\Delta \Psi$"},
    "delta_pH_inVolts": {"color": "#fa323a", "ls": "--", "label": r"$\Delta$pH"},
    "pmf": {"color": "#0d6806", "ls": "-", "label": r"$pmf$"},
    "LEF": {"color": "#157814", "ls": "--"},
    "singO2": {"color": "#fb131f", "ls": "-"},
}

for sim_ver, df in fig4_res.groupby(level="sim_ver"):
    res = df.droplevel("sim_ver")
    res.index = res.index / 60 / 60

    if sim_ver == "normal_sin":
        col_idx = 0
    elif sim_ver == "fast_sin":
        col_idx = 1
    elif sim_ver == "normal_square":
        col_idx = 2
    elif sim_ver == "fast_square":
        col_idx = 3

    for row_idx in range(len(axs)):
        tmp_ax = axs[row_idx, col_idx].twinx()
        tmp_ax.set_yticks([])
        if "sin" in sim_ver:
            xvals = np.linspace(0, 60 * 60, 100)
            tmp_ax.fill_between(
                xvals / 60 / 60, [sin_wave(x) for x in xvals], 0, color="red", alpha=0.2
            )
            tmp_ax.set_ylim(0, 300)
        else:
            x = 2.5 / 60
            while x < 1:
                tmp_ax.add_patch(
                    plt.Rectangle(
                        xy=(x, 0), width=2.5 / 60, height=1, facecolor="red", alpha=0.2
                    )
                )
                x += 5 / 60
            tmp_ax.set_ylim(0, 1)

    for var in ["Dpsi", "delta_pH_inVolts", "pmf"]:
        axs[0, col_idx].plot(res[var] - res[var].iloc[0], **plot_stylings.get(var, {}))

    axs[1, col_idx].plot(res["LEF"] * 1e-5, **plot_stylings.get("LEF", {}))
    ax_O2 = axs[1, col_idx].twinx()
    ax_O2.plot(res["singO2"] * 1e-4, **plot_stylings.get("singO2", {}))

    axs[0, col_idx].set_ylim(-0.05, 0.15)
    axs[0, col_idx].set_yticks(np.linspace(0, 0.10, 3))
    axs[0, col_idx].axhline(y=0, color="black", ls="--", alpha=0.5)
    axs[0, 0].set_ylabel(r"$\Delta$V")

    axs[1, col_idx].set_ylim([0, 3.33])
    axs[1, col_idx].set_yticks(np.linspace(0, 3, 4))
    axs[1, 0].set_ylabel(
        "LEF cumulative", color=plot_stylings.get("LEF", {}).get("color", "black")
    )
    axs[1, col_idx].spines["left"].set_color(
        plot_stylings.get("LEF", {}).get("color", "black")
    )
    axs[1, col_idx].spines["right"].set_visible(False)
    axs[1, col_idx].tick_params(
        axis="y", colors=plot_stylings.get("LEF", {}).get("color", "black")
    )
    axs[1, 0].text(
        0,
        1.01,
        r"1 $\times 10^5$",
        transform=axs[1, 0].transAxes,
        color=plot_stylings.get("LEF", {}).get("color", "black"),
    )
    ax_O2.set_ylim([0, 2 + 0.5 / 4])
    ax_O2.set_yticks(np.linspace(0, 2, 5))
    ax_O2.set_ylabel(
        r"$^1\mathrm{O}_2$ (cumulative)",
        color=plot_stylings.get("singO2", {}).get("color", "black"),
    )
    ax_O2.spines["right"].set_color(
        plot_stylings.get("singO2", {}).get("color", "black")
    )
    ax_O2.spines["left"].set_visible(False)
    ax_O2.tick_params(
        axis="y", colors=plot_stylings.get("singO2", {}).get("color", "black")
    )
    if col_idx == len(axs[0]) - 1:
        ax_O2.text(
            1,
            1.01,
            r"1 $\times 10^4$",
            transform=ax_O2.transAxes,
            color=plot_stylings.get("singO2", {}).get("color", "black"),
            ha="right",
        )

    for row_idx in range(len(axs)):
        ax = axs[row_idx, col_idx]

        if col_idx != len(axs[0]) - 1:
            for sec_ax in [ax_O2]:
                sec_ax.set_yticks([])
                sec_ax.set_ylabel("")

        if col_idx != 0:
            ax.set_yticks([])
            ax.set_ylabel("")

        ax.text(
            0.85,
            0.15,
            f"${string.ascii_lowercase[col_idx]}.{row_idx + 1}$",
            ha="center",
            va="center",
            transform=ax.transAxes,
            bbox={
                "boxstyle": "circle,pad=0.4",
                "edgecolor": "black",
                "facecolor": "white",
            },
        )

    axs[1, col_idx].set_xlim(0, 1)
    axs[1, col_idx].set_xticks(np.linspace(0, 1, 3))
    axs[1, col_idx].set_xlabel("time (h)")

axs[0, 0].legend(frameon=False)

plt.tight_layout()
plt.show()

## Figure 5

In [ ]:
def fig5_sim(settings: dict):
    m = get_davis2017()
    m.update_variables(settings["y0"])
    m.update_parameter("P_K", settings["P_K"])

    s = Simulator(m)
    s.simulate_protocol(settings["protocol"], time_points_per_step=int(1e4))

    return s.get_result().unwrap_or_err().get_combined()


fig5_prtc = make_protocol(
    [
        (2.5 * 60, {"PPFD": 0}),
        (5 * 60, {"PPFD": 300}),
        (10 * 60, {"PPFD": 0}),
    ]
)

settings = {
    "0": {
        "y0": {
            "QA_red": 1.6680442448215374e-05,
            "PQH_2": 0.01996349863700503,
            "pH_lumen": 6.455422734896244,
            "Dpsi": 0.014062169675170863,
            "K_lu": 0.023317976223034567,
            "PC_ox": 1.2976782989183835,
            "Zx": 0.07361151276346266,
            "singO2": 0.2874820769637723,
            "P700_ox": 0,
            "Fd_red": 0,
            "NADPH_st": 0,
            "LEF": 0,
            "ATP_made": 0,
        },
        "P_K": get_davis2017().get_parameter_values()["P_K"] * 0,
        "protocol": fig5_prtc,
    },
    "1": {
        "y0": {
            "QA_red": 5.511496362768013e-06,
            "PQH_2": 0.006610917859794705,
            "pH_lumen": 6.455489733520293,
            "Dpsi": 0.014066213897372587,
            "K_lu": 0.023314403407362544,
            "PC_ox": 1.6965704506492556,
            "Zx": 0.0735694070105809,
            "singO2": 0.38023965479069094,
            "P700_ox": 0,
            "Fd_red": 0,
            "NADPH_st": 0,
            "LEF": 0,
            "ATP_made": 0,
        },
        "P_K": get_davis2017().get_parameter_values()["P_K"] * 1,
        "protocol": fig5_prtc,
    },
    "10": {
        "y0": {
            "QA_red": 1.6680442448215374e-05,
            "PQH_2": 0.01996349863700503,
            "pH_lumen": 6.455422734896244,
            "Dpsi": 0.014062169675170863,
            "K_lu": 0.023317976223034567,
            "PC_ox": 1.2976782989183835,
            "Zx": 0.07361151276346266,
            "singO2": 0.2874820769637723,
            "P700_ox": 0,
            "Fd_red": 0,
            "NADPH_st": 0,
            "LEF": 0,
            "ATP_made": 0,
        },
        "P_K": get_davis2017().get_parameter_values()["P_K"] * 10,
        "protocol": fig5_prtc,
    },
    "100": {
        "y0": {
            "QA_red": 1.6680442448215374e-05,
            "PQH_2": 0.01996349863700503,
            "pH_lumen": 6.455422734896244,
            "Dpsi": 0.014062169675170863,
            "K_lu": 0.023317976223034567,
            "PC_ox": 1.2976782989183835,
            "Zx": 0.07361151276346266,
            "singO2": 0.2874820769637723,
            "P700_ox": 0,
            "Fd_red": 0,
            "NADPH_st": 0,
            "LEF": 0,
            "ATP_made": 0,
        },
        "P_K": 600000,
        "protocol": fig5_prtc,
    },
}

fig5_res = dict(parallelise(fig5_sim, list(settings.items())))
fig5_res = pd.concat(fig5_res.values(), keys=fig5_res.keys())
fig5_res.index.names = ["P_K_fold", "time"]

In [ ]:
fig5, axs = plt.subplots(ncols=3)

time_start = fig5_prtc.iloc[0].name.total_seconds()

plot_stylings = {
    "0": {
        "color": "red",
    },
    "1": {
        "color": "orange",
    },
    "10": {
        "color": "green",
    },
    "100": {
        "color": "blue",
    },
}

for fold, df in fig5_res.groupby(level="P_K_fold"):
    res = df.droplevel("P_K_fold")
    res = res[res.index >= time_start]
    res.index = res.index - time_start

    axs[0].plot(res["v_VKC"], **plot_stylings[fold], label=f"P = {fold}X")

    dt = pd.Series(res.index).diff().values
    res["LEF_to_NADPH"] = res["LEF"].diff() / dt
    res["ATP_rate"] = res["ATP_made"].diff() / dt
    res["LEF_cumulative"] = (res["LEF"] * dt).fillna(0).cumsum()
    res["deficit"] = (res["LEF_to_NADPH"] * (3.0 / 4.666) - res["ATP_rate"]).fillna(0)
    res["deficit_int"] = scipy.integrate.cumulative_trapezoid(
        res["deficit"], res.index.astype(float), initial=0
    )

    axs[1].plot(res["deficit"] * 5, **plot_stylings[fold], label=f"P = {fold}X")
    axs[2].plot(res["deficit_int"] / 2, **plot_stylings[fold], label=f"P = {fold}X")

for i, ax in enumerate(axs):
    ax.set_xlim(-1, 30)
    ax.set_xlabel("time (s)")

    ax.text(
        0.85,
        0.15,
        f"${string.ascii_lowercase[i]}$",
        ha="center",
        va="center",
        transform=ax.transAxes,
        bbox={"boxstyle": "circle,pad=0.4", "edgecolor": "black", "facecolor": "white"},
    )

axs[0].set_ylim(-25, 425)
axs[0].set_yticks(np.linspace(0, 400, 9))
axs[0].set_ylabel("K$^+$ flux (mol mol $^{-1}$ PSII s$^{-1}$)")

axs[1].set_ylim(-25, 425)
axs[1].set_yticks(np.linspace(0, 400, 9))
axs[1].set_ylabel("proton deficient (fraction)")

axs[2].set_ylim(-10, 150)
axs[2].set_yticks(np.linspace(0, 140, 8))
axs[2].set_ylabel("ATP deficient (cumulative)")

axs[1].legend(frameon=False)

plt.tight_layout()
plt.show()